# Hafta 12 — Yapay Sinir Ağlarına Giriş

> Hücreleri **yukarıdan aşağıya sırayla** çalıştırın; sonraki hücreler önceki hücrelerde tanımlanan değişken ve fonksiyonları kullanır. Tek başına çalışmayan gösterim amaçlı kod parçaları markdown olarak bırakılmıştır. Veri setleri ilk çalıştırmada internetten indirilir.

In [ ]:
# Gerekli paketler kurulu değilse aşağıdaki satırın başındaki # işaretini kaldırıp çalıştırın
# %pip install numpy pandas matplotlib seaborn scikit-learn scipy torch torchvision

## Basit Anlatım: Bu Hafta Ne Öğreneceğiz?

Yapay sinir ağları, beyindeki sinir hücrelerinden esinlenmiştir. Her yapay sinir hücresi, kendisine gelen bilgileri önem derecelerine göre birleştirip bir sonraki katmana iletir. Binlerce hücre katmanlar halinde birbirine bağlandığında çok karmaşık ilişkileri öğrenebilen bir yapı ortaya çıkar.

Ağ yanlış bir tahmin yaptığında hata sondan başa doğru geri gönderilir. Her bağlantı, bu hataya ne kadar katkı yaptıysa o oranda düzeltilir. Bunu, kulağına yanlış gelen sesi kimin çıkardığını bulup o müzisyenleri uyaran bir orkestra şefine benzetebilirsiniz.

---

## Bu Haftanın Hedefleri

- Yapay nöronu ve perceptronun sınırlarını anlamak
- Çok katmanlı ağların ve aktivasyon fonksiyonlarının rolünü kavramak
- Geri yayılımı küçük bir ağda sayılarla izlemek
- PyTorch ile bir sinir ağı eğitmek

---

## 1. Yapay Nöron ve Perceptron

```
  x₁ ──w₁──╮
  x₂ ──w₂──┼──► z = w₁x₁ + w₂x₂ + w₃x₃ + b ──► f(z) ──► çıktı
  x₃ ──w₃──╯
```

Nöron, girdileri ağırlıklarıyla çarpıp toplar ve sonucu bir **aktivasyon fonksiyonundan** geçirir. f sigmoid seçilirse bu, Hafta 5'teki lojistik regresyonun ta kendisidir.

**Perceptron** en basit nörondur: z > 0 ise 1, değilse 0 çıktısı verir. Yanlış tahmin yaptığında ağırlıklarını şöyle günceller:

```
w ← w + η·(y − ŷ)·x
```

Perceptron VE (AND) kapısını öğrenebilir. **XOR'u ise öğrenemez**, çünkü XOR'un iki sınıfı tek bir düz çizgiyle ayrılamaz:

```
 x₂
  1 │ ●(0,1)=1     ○(1,1)=0
  0 │ ○(0,0)=0     ●(1,0)=1      Tek bir doğruyla ● ve ○ ayrılamaz
    └─────────────────── x₁
```

Çözüm, **birden fazla katman** kullanmaktır.

---

## 2. Çok Katmanlı Ağ (MLP)

```
 Girdi          Gizli katman(lar)          Çıktı
   x₁ ──────── ● ──────── ● ──────────── ŷ₁
   x₂ ──╳───── ● ──╳───── ● ──╳───────── ŷ₂
   x₃ ──────── ● ──────── ● ──────────── ŷ₃
```

Her katmanda aynı iki işlem yapılır: `z = W·a + b` ve `a = f(z)`. Gizli katmanlar, girdiyi problemin daha kolay çözülebildiği yeni bir biçime dönüştürmeyi kendileri öğrenir.

### Aktivasyon fonksiyonları neden gerekli?

Aktivasyon olmasaydı, kaç katman olursa olsun ağ tek bir doğrusal modele indirgenirdi. Doğrusal olmayan aktivasyonlar ağa eğrisel ilişkileri öğrenme gücü verir.

| Fonksiyon | Tanım | Kullanım |
|---|---|---|
| **ReLU** | max(0, z) | Gizli katmanlarda varsayılan seçim |
| **Sigmoid** | 1 / (1 + e⁻ᶻ) | İkili sınıflandırma çıktısı |
| **Softmax** | Skorları toplamı 1 olan olasılıklara çevirir | Çok sınıflı çıktı |

Sigmoid'in türevi en fazla 0.25'tir. Derin bir ağda bu küçük sayılar katman katman çarpıldıkça ilk katmanlara ulaşan sinyal neredeyse sıfırlanır (**kaybolan gradyan**). ReLU'nun türevi pozitif bölgede 1 olduğu için bu sorun yaşanmaz.

---

## 3. Geri Yayılım: Sayılarla Bir Adım

**Ağ:** 2 girdi → 2 gizli nöron (sigmoid) → 1 çıktı (sigmoid). Girdi x = (1, 2), gerçek etiket y = 1, öğrenme oranı η = 0.5.

Başlangıç ağırlıkları: h₁ için (0.1, 0.2), h₂ için (−0.3, 0.1), çıktı için v = (0.5, −0.4). Bütün bias değerleri 0.

**İleri yayılım:**

```
h₁ = σ(0.1·1 + 0.2·2)   = σ(0.5)   = 0.622
h₂ = σ(−0.3·1 + 0.1·2)  = σ(−0.1)  = 0.475
ŷ  = σ(0.5·0.622 − 0.4·0.475) = σ(0.121) = 0.530
Kayıp = −ln(0.530) = 0.634
```

**Geri yayılım** (zincir kuralıyla hata sinyalleri):

```
Çıktı hatası:   δ₃ = ŷ − y = −0.470
Gizli hatalar:  δ₁ = δ₃ · v₁ · h₁(1 − h₁) = −0.055
                δ₂ = δ₃ · v₂ · h₂(1 − h₂) = +0.047
```

Her ağırlığın gradyanı = (ilgili hata sinyali) × (o ağırlığa gelen girdi). Örneğin v₁'in gradyanı δ₃ · h₁ = −0.292 olur ve v₁ şöyle güncellenir: 0.5 − 0.5 × (−0.292) = **0.646**.

**Tüm ağırlıklar güncellendikten sonra:** ŷ = 0.631 ve kayıp = 0.461 olur. Kayıp **0.634'ten 0.461'e düştü**; tahmin doğru cevaba (1) yaklaştı.

In [ ]:
import numpy as np

def sigmoid(z): return 1 / (1 + np.exp(-z))

x, y = np.array([1.0, 2.0]), 1.0
W1, b1 = np.array([[0.1, 0.2], [-0.3, 0.1]]), np.zeros(2)
v, c = np.array([0.5, -0.4]), 0.0

for adim in range(3):
    h = sigmoid(W1 @ x + b1)                       # ileri yayılım
    y_hat = sigmoid(v @ h + c)
    d3 = y_hat - y                                 # geri yayılım
    d_h = d3 * v * h * (1 - h)
    v, c = v - 0.5 * d3 * h, c - 0.5 * d3          # güncelleme
    W1, b1 = W1 - 0.5 * np.outer(d_h, x), b1 - 0.5 * d_h
    print(f"Adım {adim}: ŷ = {y_hat:.3f}, kayıp = {-np.log(y_hat):.3f}")

PyTorch gibi kütüphaneler bu hesabı `backward()` komutuyla otomatik yapar.

---

## 4. Eğitimi İyileştirmek

**Optimizasyon algoritmaları:**

| Yöntem | Fikir |
|---|---|
| **SGD** | Küçük veri grupları (mini-batch) üzerinden gradyan iniş |
| **Momentum** | Önceki adımların yönünü hatırlayarak ivme kazanır, zigzagı azaltır |
| **Adam** | Momentum + her parametre için kendi kendini ayarlayan öğrenme oranı; en yaygın seçim |

**Dropout:** Eğitim sırasında her adımda nöronların bir kısmı rastgele kapatılır. Böylece ağ, belirli nöronlara bağımlı kalmadan daha sağlam özellikler öğrenir. Test sırasında bütün nöronlar açıktır.

**Erken durdurma:** Doğrulama hatası artık iyileşmiyorsa eğitim durdurulur.

---

## 5. Laboratuvar: PyTorch ile MLP

**Veri:** Fashion-MNIST. 10 sınıf giyim ürünü içerir (tişört, pantolon, ayakkabı...) ve görüntüler 28×28 piksel boyutundadır.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(0)
donusum = transforms.ToTensor()
egitim = datasets.FashionMNIST("veri", train=True, download=True, transform=donusum)
test = datasets.FashionMNIST("veri", train=False, download=True, transform=donusum)
egitim_y = DataLoader(egitim, batch_size=128, shuffle=True)
test_y = DataLoader(test, batch_size=512)

model = nn.Sequential(
    nn.Flatten(),                                    # 28×28 → 784
    nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(128, 10),                              # 10 sınıf (softmax kayıp içinde)
)
kayip_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    model.train()
    for X, y in egitim_y:
        kayip = kayip_fn(model(X), y)     # 1) ileri yayılım ve kayıp
        optimizer.zero_grad()             # 2) eski gradyanları sıfırla
        kayip.backward()                  # 3) geri yayılım
        optimizer.step()                  # 4) ağırlıkları güncelle

    model.eval()                          # dropout kapalı
    with torch.no_grad():
        dogru = sum((model(X).argmax(1) == y).sum().item() for X, y in test_y)
    print(f"Epoch {epoch + 1}: test doğruluğu = {dogru / len(test):.3f}")   # ~%87–88

**Görevler:**
1. Adam yerine `torch.optim.SGD(model.parameters(), lr=0.01)` kullanın. Öğrenme hızı nasıl değişti?
2. Dropout oranını 0 ve 0.6 yapıp eğitim ve test doğruluklarını karşılaştırın.
3. Gizli katmanlardaki ReLU'ları Sigmoid ile değiştirin ve sonucu yorumlayın.

---

## 6. Özet

- Nöron, girdilerin ağırlıklı toplamını bir aktivasyon fonksiyonundan geçirir.
- Tek katman XOR'u çözemez; çok katmanlı ağlar karmaşık ilişkileri öğrenir.
- Geri yayılım, hata sinyalini çıktıdan girdiye taşıyarak bütün ağırlıkların gradyanını hesaplar.
- Adam, dropout ve erken durdurma eğitimi hızlandırır ve aşırı öğrenmeyi azaltır.

## Kendinizi Sınayın

1. 100 → 50 → 10 ağında kaç parametre (ağırlık + bias) vardır?
2. Bütün ağırlıkları sıfırla başlatırsak ne olur?
3. Dropout neden test sırasında kapatılır?

## Çalışma Alanı

Alıştırmalar ve "Kendinizi Sınayın" soruları için aşağıdaki hücreleri kullanın.